# XGBoost

Train and tune an XGBoost classifier for flight cancellation prediction. We run on both feature sets (with and without `IS_COVID`) and compare.

For each version:
1. Train a baseline XGBoost with early stopping on validation PR-AUC
2. Run a randomised hyperparameter search
3. Pick the best config, retrain on full training data
4. Tune the decision threshold on validation (optimise F2)
5. Report final metrics on the test set

XGBoost handles early stopping natively via `eval_set`, so we don't need a separate retraining step — the search already uses the best iteration.

In [3]:
import os
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

MODEL_DIR = 'model_weights'
os.makedirs(MODEL_DIR, exist_ok=True)

## Load the data

Both feature versions from the single `artifacts/` directory. Targets are shared.

In [4]:
datasets = {}
for version in ['with_covid', 'no_covid']:
    datasets[version] = {
        'X_train': pd.read_parquet(f'artifacts/X_train_{version}.parquet'),
        'X_val':   pd.read_parquet(f'artifacts/X_val_{version}.parquet'),
        'X_test':  pd.read_parquet(f'artifacts/X_test_{version}.parquet'),
    }

y_train = pd.read_parquet('artifacts/y_train.parquet')['CANCELLED'].values
y_val   = pd.read_parquet('artifacts/y_val.parquet')['CANCELLED'].values
y_test  = pd.read_parquet('artifacts/y_test.parquet')['CANCELLED'].values

# XGBoost uses this to weight the loss for the minority class
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f'Train: {len(y_train):,} rows  |  cancel rate: {y_train.mean():.2%}')
print(f'Val:   {len(y_val):,} rows  |  cancel rate: {y_val.mean():.2%}')
print(f'Test:  {len(y_test):,} rows  |  cancel rate: {y_test.mean():.2%}')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')
print(f'\nWith COVID: {datasets["with_covid"]["X_train"].shape[1]} features')
print(f'No COVID:   {datasets["no_covid"]["X_train"].shape[1]} features')

Train: 2,186,803 rows  |  cancel rate: 2.92%
Val:   349,591 rows  |  cancel rate: 2.16%
Test:  698,587 rows  |  cancel rate: 2.06%
scale_pos_weight: 33.3

With COVID: 25 features
No COVID:   24 features


## Approach

XGBoost has a lot of knobs. Instead of manually picking 3 configs like the original notebooks did, we run `RandomizedSearchCV` over a broad grid. The search uses `scoring='average_precision'` (PR-AUC) with 3-fold stratified CV.

After finding the best hyperparameters, we retrain on the full training set with early stopping on validation, then sweep thresholds for F2.

In [5]:
def find_best_threshold(y_true, y_proba, beta=2):
    """Sweep thresholds in [0, 1] and return the one that maximises F-beta."""
    best_t, best_score = 0.5, 0.0
    for t in np.linspace(0, 1, 101):
        score = fbeta_score(y_true, (y_proba >= t).astype(int), beta=beta, zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score


def evaluate_on_test(model, X_test, y_test, threshold):
    """Get PR-AUC and F2 on the test set at a given threshold."""
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= threshold).astype(int)
    return {
        'PR-AUC': average_precision_score(y_test, proba),
        'F2': fbeta_score(y_test, preds, beta=2, zero_division=0),
        'Threshold': threshold,
    }

## Train and tune

For each feature version we train a baseline, then search over a broad grid. The grid covers tree depth, learning rate, regularisation (gamma, L1, L2), subsampling, and different class weight multipliers.

We use `tree_method='hist'` with `device='cuda'` for GPU-accelerated training. If no GPU is available, change `device='cuda'` to `device='cpu'`.

In [6]:
param_grid = {
    'max_depth': [4, 5, 6, 8],
    'learning_rate': [0.02, 0.03, 0.05, 0.1],
    'n_estimators': [1000, 1500, 2500],
    'min_child_weight': [3, 5, 8, 10],
    'subsample': [0.7, 0.75, 0.8],
    'colsample_bytree': [0.7, 0.75, 0.8],
    'gamma': [0, 0.5, 1.0],
    'reg_alpha': [0, 0.05, 0.1],
    'reg_lambda': [1.0, 2.0, 2.5],
    'scale_pos_weight': [
        scale_pos_weight * 0.8,
        scale_pos_weight,
        scale_pos_weight * 1.2,
    ],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
all_results = []

for version in ['with_covid', 'no_covid']:
    X_tr = datasets[version]['X_train']
    X_v  = datasets[version]['X_val']
    X_te = datasets[version]['X_test']

    print(f'\n{"=" * 60}')
    print(f'  {version}  ({X_tr.shape[1]} features)')
    print(f'{"=" * 60}')

    # --- Baseline: default XGBoost with early stopping ---
    print('\n[1] Baseline XGBoost...')
    xgb_base = XGBClassifier(
        objective='binary:logistic', eval_metric='aucpr',
        tree_method='hist', device='cuda',
        scale_pos_weight=scale_pos_weight, max_depth=6,
        learning_rate=0.1, n_estimators=1000,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, early_stopping_rounds=50)
    xgb_base.fit(X_tr, y_train, eval_set=[(X_v, y_val)], verbose=False)

    val_proba = xgb_base.predict_proba(X_v)[:, 1]
    base_t, base_val_f2 = find_best_threshold(y_val, val_proba)
    base_test = evaluate_on_test(xgb_base, X_te, y_test, base_t)
    print(f'    Best iteration: {xgb_base.best_iteration}')
    print(f'    Val F2: {base_val_f2:.4f} (threshold={base_t:.2f})')
    print(f'    Test PR-AUC: {base_test["PR-AUC"]:.4f}  |  Test F2: {base_test["F2"]:.4f}')

    all_results.append({
        'Version': version, 'Model': 'Baseline XGBoost', **base_test})

    # --- Hyperparameter search ---
    print('\n[2] RandomizedSearchCV (80 iterations, 3-fold CV)...')
    print('    This will take a while.')

    search = RandomizedSearchCV(
        XGBClassifier(
            objective='binary:logistic', eval_metric='aucpr',
            tree_method='hist', device='cuda',
            random_state=42, n_jobs=-1),
        param_grid,
        n_iter=80,
        scoring='average_precision',
        cv=cv,
        random_state=42,
        verbose=1,
    )
    search.fit(X_tr, y_train)

    print(f'    Best CV PR-AUC: {search.best_score_:.4f}')
    print(f'    Best params: {search.best_params_}')

    # --- Retrain best config with early stopping on validation ---
    print('\n[3] Retraining best config with early stopping...')
    best_params = search.best_params_.copy()
    best_params['objective'] = 'binary:logistic'
    best_params['eval_metric'] = 'aucpr'
    best_params['tree_method'] = 'hist'
    best_params['device'] = 'cuda'
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    best_params['early_stopping_rounds'] = 50
    if 'n_estimators' not in best_params:
        best_params['n_estimators'] = 2000

    best_xgb = XGBClassifier(**best_params)
    best_xgb.fit(X_tr, y_train, eval_set=[(X_v, y_val)], verbose=False)

    val_proba = best_xgb.predict_proba(X_v)[:, 1]
    tuned_t, tuned_val_f2 = find_best_threshold(y_val, val_proba)
    tuned_test = evaluate_on_test(best_xgb, X_te, y_test, tuned_t)
    print(f'    Best iteration: {best_xgb.best_iteration}')
    print(f'    Val F2: {tuned_val_f2:.4f} (threshold={tuned_t:.2f})')
    print(f'    Test PR-AUC: {tuned_test["PR-AUC"]:.4f}  |  Test F2: {tuned_test["F2"]:.4f}')

    all_results.append({
        'Version': version, 'Model': 'Tuned XGBoost', **tuned_test})

    # --- Save the tuned model ---
    out_path = f'{MODEL_DIR}/xgboost_{version}.pkl'
    joblib.dump(best_xgb, out_path)
    print(f'\n    Saved: {out_path}')


  with_covid  (25 features)

[1] Baseline XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [05:10:24] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


    Best iteration: 87
    Val F2: 0.2201 (threshold=0.51)
    Test PR-AUC: 0.0686  |  Test F2: 0.1800

[2] RandomizedSearchCV (80 iterations, 3-fold CV)...
    This will take a while.
Fitting 3 folds for each of 80 candidates, totalling 240 fits
    Best CV PR-AUC: 0.3817
    Best params: {'subsample': 0.8, 'scale_pos_weight': np.float64(33.28343210108801), 'reg_lambda': 2.5, 'reg_alpha': 0, 'n_estimators': 2500, 'min_child_weight': 10, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.5, 'colsample_bytree': 0.8}

[3] Retraining best config with early stopping...
    Best iteration: 86
    Val F2: 0.2071 (threshold=0.47)
    Test PR-AUC: 0.0692  |  Test F2: 0.1790

    Saved: model_weights/xgboost_with_covid.pkl

  no_covid  (24 features)

[1] Baseline XGBoost...
    Best iteration: 134
    Val F2: 0.2316 (threshold=0.47)
    Test PR-AUC: 0.0485  |  Test F2: 0.1335

[2] RandomizedSearchCV (80 iterations, 3-fold CV)...
    This will take a while.
Fitting 3 folds for each of 80 candidate

## Results

Baseline vs tuned, and with_covid vs no_covid — did the tuning help, and does the COVID flag make a difference?

In [7]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

   Version            Model   PR-AUC       F2  Threshold
with_covid Baseline XGBoost 0.068612 0.179970       0.51
with_covid    Tuned XGBoost 0.069226 0.178997       0.47
  no_covid Baseline XGBoost 0.048529 0.133484       0.47
  no_covid    Tuned XGBoost 0.043535 0.124502       0.49
